In [ ]:
%pip install git+https://github.com/balaji1810/MLSec-Transfer-Attacks.git@pip -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.1/116.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.8/54.8 kB 3.5 MB/s eta 0:00:00


In [2]:
import copy
import json
import os
import time

import numpy as np
import torch
import torchvision
import yaml

from src.data import load_cifar10_testset
from src.models import load_surrogate, load_target, build_ensemble
from src.attacks import create_attack, generate_adversarial_examples
from src.evaluate import evaluate_transfer
from src.metadata import get_reported_robust_acc, get_reported_clean_acc
from src.reporting import generate_all_reports, save_fooled_images


In [ ]:
CIFAR10_CLASSES = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck",
]

_DEFAULT_CONFIG = {
    "seed": 42,
    "device": "auto",
    "data": {
        "dataset": "cifar10",
        "data_dir": "/kaggle/input/datasets/balajisb18/cifar-10",
        "num_samples": None,
        "batch_size": 5,
    },
    "threat_model": {
        "norm": "Linf",
        "epsilon": 8 / 255,
    },
    "enable_single_surrogate_attacks": False,
    "surrogates": [
        {
            "name": "Standard",
            "source": "robustbench",
            "needs_normalize": False,
        },
        {
            "name": "cifar10_vgg16_bn",
            "source": "chenyaofo",
            "needs_normalize": True,
        },
        {
            "name": "cifar10_resnet56",
            "source": "chenyaofo",
            "needs_normalize": True,
        },
        {
            "name": "cifar10_mobilenetv2_x1_4",
            "source": "chenyaofo",
            "needs_normalize": True,
        },
        {
            "name": "cifar10_repvgg_a2",
            "source": "chenyaofo",
            "needs_normalize": True,
        },
    ],
    "targets": [
        {
            "name": "Kang2021Stable",
            "source": "robustbench",
            "needs_normalize": False
        },
        {
            "name": "Rice2020Overfitting",
            "source": "robustbench",
            "needs_normalize": False
        },
        {
            "name": "Wong2020Fast",
            "source": "robustbench",
            "needs_normalize": False
        },
        {
            "name": "Andriushchenko2020Understanding",
            "source": "robustbench",
            "needs_normalize": False
        },
        {
            "name": "Ding2020MMA",
            "source": "robustbench",
            "needs_normalize": False
        },
        {
            "name": "Engstrom2019Robustness",
            "source": "robustbench",
            "needs_normalize": False
        },
        {
            "name": "Zhang2019You",
            "source": "robustbench",
            "needs_normalize": False
        },
        {
            "name": "Sitawarin2020Improving",
            "source": "robustbench",
            "needs_normalize": False
        },
        {
            "name": "Huang2020Self",
            "source": "robustbench",
            "needs_normalize": False
        },
        {
            "name": "Chen2020Efficient",
            "source": "robustbench",
            "needs_normalize": False
        },
        {
            "name": "Chen2020Adversarial",
            "source": "robustbench",
            "needs_normalize": False
        },
        {
            "name": "Zhang2019Theoretically",
            "source": "robustbench",
            "needs_normalize": False
        }
    ],
    "attacks": [
        # {
        #     "name": "MIFGSM",
        #     "params": {
        #         "steps": 10,
        #         "decay": 1.0
        #     }
        # },
        # {
        #     "name": "DIFGSM",
        #     "params": {
        #         "steps": 10,
        #         "decay": 1.0,
        #         "resize_rate": 0.9,
        #         "diversity_prob": 1.0
        #     }
        # },
        # {
        #     "name": "TIFGSM",
        #     "params": {
        #         "steps": 10,
        #         "decay": 1.0,
        #         "kern_len": 15,
        #         "n_sig": 3
        #     }
        # },
        # {
        #     "name": "SINIFGSM",
        #     "params": {
        #         "steps": 15,
        #         "decay": 1.0,
        #         "m": 3
        #     }
        # },
        {
            "name": "Admix",
            "params": {
                "steps": 15,
                "decay": 1.0,
                "portion": 0.2,
                "size": 3,
                "num_classes": 10
            }
        },
        # {
        #     "name": "VMIFGSM",
        #     "params": {
        #         "steps": 10,
        #         "decay": 1.0,
        #         "n": 5,
        #         "beta": 1.5
        #     }
        # },
    ],
    "ensemble": {
        "enabled": True,
    },
    "output": {
        "results_dir": "/kaggle/working/results",
        "save_csv": True,
        "save_plots": True,
        "save_adv_examples": True,
    },
    "model_dir": "/kaggle/input/models/balajisb18/cifar-10/pytorch/default/1/models",
}

In [4]:
def get_default_config() -> dict:
    """Return the default experiment configuration as a mutable dict.

    Callers can modify the returned dict freely without affecting the defaults.
    """
    return copy.deepcopy(_DEFAULT_CONFIG)


def load_config() -> dict:
    """Load a YAML configuration file."""
    # with open(path, "r") as f:
    #     return yaml.safe_load(f)
    return copy.deepcopy(_DEFAULT_CONFIG)


def set_seed(seed: int) -> None:
    """Set random seeds for reproducibility."""
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def get_device(device_str: str) -> torch.device:
    """Resolve device string to torch.device."""
    if device_str == "auto":
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    return torch.device(device_str)

In [5]:
def run_single_experiment(
    surrogate_name: str,
    surrogate_model: torch.nn.Module,
    normalize_fn,
    attack_name: str,
    attack_params: dict,
    target_configs: list[dict],
    images: torch.Tensor,
    labels: torch.Tensor,
    eps: float,
    batch_size: int,
    model_dir: str,
    device: torch.device,
    output_dir: str,
    save_fooled: bool = True,
) -> list[dict]:
    print(f"\n{'-' * 60}")
    print(f"Surrogate: {surrogate_name}  |  Attack: {attack_name}")
    print(f"{'-' * 60}")

    # Create attack
    attack = create_attack(
        attack_name=attack_name,
        model=surrogate_model,
        normalize=normalize_fn,
        device=device,
        eps=eps,
        **attack_params,
    )

    # Generate adversarial examples on surrogate
    t0 = time.time()
    adv_images = generate_adversarial_examples(
        attack=attack,
        images=images,
        labels=labels,
        batch_size=batch_size,
        device=device,
    )
    gen_time = time.time() - t0
    print(f"Adversarial examples generated in {gen_time:.1f}s")

    # Evaluate transfer to each target
    results = []
    for tgt_cfg in target_configs:
        tgt_name = tgt_cfg["name"]
        print(f"Evaluating transfer to {tgt_name}")

        target_model = load_target(tgt_cfg, model_dir=model_dir, device=device)

        metrics = evaluate_transfer(
            target_model=target_model,
            clean_images=images,
            adv_images=adv_images,
            labels=labels,
            batch_size=batch_size,
            device=device,
        )

        # Save fooled adversarial images
        fooled_mask = metrics.pop("fooled_mask")
        if fooled_mask.any() and save_fooled:
            adv_preds = metrics.pop("adv_preds")
            clean_preds = metrics.pop("clean_preds")
            save_fooled_images(
                clean_images=images[fooled_mask],
                adv_images=adv_images[fooled_mask],
                labels=labels[fooled_mask],
                clean_preds=clean_preds[fooled_mask],
                adv_preds=adv_preds[fooled_mask],
                surrogate_name=surrogate_name,
                attack_name=attack_name,
                target_name=tgt_name,
                output_dir=os.path.join(output_dir, "fooled_images"),
            )

        # Reported values
        reported_robust = get_reported_robust_acc(tgt_name)
        reported_clean = get_reported_clean_acc(tgt_name)

        # Delta: how much lower our transfer attack acc is vs reported robust acc
        # Negative delta means our attack is "stronger" than reported AutoAttack
        delta = None
        if reported_robust is not None:
            delta = round(metrics["adversarial_accuracy"] - reported_robust, 2)

        result = {
            "surrogate": surrogate_name,
            "attack": attack_name,
            "target": tgt_name,
            "num_samples": metrics["num_samples"],
            "clean_accuracy": metrics["clean_accuracy"],
            "adversarial_accuracy": metrics["adversarial_accuracy"],
            "attack_success_rate": metrics["attack_success_rate"],
            "mean_linf_perturbation": metrics["mean_linf_perturbation"],
            "max_linf_perturbation": metrics["max_linf_perturbation"],
            "reported_clean_acc": reported_clean,
            "reported_robust_acc": reported_robust,
            "delta_vs_reported": delta,
            "generation_time_s": round(gen_time, 1),
        }
        results.append(result)

        print(f"Clean acc: {metrics['clean_accuracy']:.2f}%  "
              f"Adv acc: {metrics['adversarial_accuracy']:.2f}%  "
              f"ASR: {metrics['attack_success_rate']:.2f}%  "
              f"Linf: {metrics['mean_linf_perturbation']:.4f}")

        # Free target model to save VRAM
        del target_model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return results


In [6]:
def _save_incremental(results: list[dict], output_dir: str) -> None:
    os.makedirs(output_dir, exist_ok=True)
    path = os.path.join(output_dir, f"results_incremental_{time.time()}.json")
    with open(path, "w") as f:
        json.dump(results, f, indent=2)

In [7]:
def main():

    # Load config
    cfg = load_config()

    seed = cfg.get("seed", 42)
    set_seed(seed)
    device = get_device(cfg.get("device", "auto"))
    eps = cfg["threat_model"]["epsilon"]
    batch_size = cfg["data"]["batch_size"]
    model_dir = cfg.get("model_dir", "/kaggle/input/models/balajisb18/cifar-10/pytorch/default/1/models")
    output_dir = cfg["output"]["results_dir"]

    print("=" * 70)
    print("Transfer-Based Adversarial Attack on CIFAR-10")
    print("=" * 70)
    print(f"Device:       {device}")
    print(f"Seed:         {seed}")
    print(f"Epsilon:      {eps} ({eps * 255:.1f}/255)")
    print(f"Num samples:  {cfg['data']['num_samples']}")
    print(f"Surrogates:   {[s['name'] for s in cfg['surrogates']]}")
    print(f"Targets:      {[t['name'] for t in cfg['targets']]}")
    print(f"Attacks:      {[a['name'] for a in cfg['attacks']]}")
    print(f"Ensemble:     {cfg['ensemble']['enabled']}")
    print(f"Models Dir:   {model_dir}")

    # Load data
    print("Loading CIFAR-10 test data")
    images, labels = load_cifar10_testset(
        data_dir=cfg["data"]["data_dir"],
        num_samples=cfg["data"]["num_samples"],
        seed=seed,
    )
    print(f"Loaded {len(labels)} samples, shape={images.shape}")

    all_results = []

    # Single-surrogate attacks
    if cfg["enable_single_surrogate_attacks"]:
        for surr_cfg in cfg["surrogates"]:
            surr_name = surr_cfg["name"]
            print(f"\n{'=' * 70}")
            print(f"Loading surrogate: {surr_name}")
            print(f"{'=' * 70}")

            surrogate_model, normalize_fn = load_surrogate(
                surr_cfg, model_dir=model_dir, device=device
            )

            for atk_cfg in cfg["attacks"]:
                atk_name = atk_cfg["name"]
                atk_params = dict(atk_cfg.get("params", {}))

                results = run_single_experiment(
                    surrogate_name=surr_name,
                    surrogate_model=surrogate_model,
                    normalize_fn=normalize_fn,
                    attack_name=atk_name,
                    attack_params=atk_params,
                    target_configs=cfg["targets"],
                    images=images,
                    labels=labels,
                    eps=eps,
                    batch_size=batch_size,
                    model_dir=model_dir,
                    device=device,
                    output_dir=output_dir,
                    save_fooled=cfg["output"]["save_adv_examples"],
                )
                all_results.extend(results)

                # Save incremental results
                # _save_incremental(all_results, output_dir)

            # Free surrogate to save VRAM
            del surrogate_model
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    # Ensemble surrogate attacks
    if cfg["ensemble"]["enabled"]:
        print(f"\n{'=' * 70}")
        print("Loading ensemble surrogate (all surrogates combined)")
        print(f"{'=' * 70}")

        ensemble_model, ensemble_norm_fn = build_ensemble(
            cfg["surrogates"], model_dir=model_dir, device=device
        )

        for atk_cfg in cfg["attacks"]:
            atk_name = atk_cfg["name"]
            atk_params = dict(atk_cfg.get("params", {}))

            results = run_single_experiment(
                surrogate_name="Ensemble",
                surrogate_model=ensemble_model,
                normalize_fn=ensemble_norm_fn,
                attack_name=atk_name,
                attack_params=atk_params,
                target_configs=cfg["targets"],
                images=images,
                labels=labels,
                eps=eps,
                batch_size=batch_size,
                model_dir=model_dir,
                device=device,
                output_dir=output_dir,
                save_fooled=cfg["output"]["save_adv_examples"],
            )
            all_results.extend(results)

            # _save_incremental(all_results, output_dir)

        del ensemble_model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # Final reports
    df = generate_all_reports(all_results, output_dir)

    # Save config snapshot
    config_snapshot_path = os.path.join(output_dir, f"config_snapshot_{seed}_{time.time()}.yaml")
    os.makedirs(output_dir, exist_ok=True)
    with open(config_snapshot_path, "w") as f:
        yaml.dump(cfg, f, default_flow_style=False)

    print(f"\n{'=' * 70}")
    print("Experiment complete")
    print(f"Results saved to: {output_dir}/")
    print(f"{'=' * 70}")


In [8]:
main()

Transfer-Based Adversarial Attack on CIFAR-10
Device:       cuda
Seed:         42
Epsilon:      0.03137254901960784 (8.0/255)
Num samples:  None
Surrogates:   ['Standard', 'cifar10_vgg16_bn', 'cifar10_resnet56', 'cifar10_mobilenetv2_x1_4', 'cifar10_repvgg_a2']
Targets:      ['Chen2020Adversarial', 'Rebuffi2021Fixing_70_16_cutmix_extra', 'Wang2023Better_WRN-70-16', 'Sehwag2021Proxy_ResNest152', 'Sehwag2021Proxy_R18', 'Addepalli2022Efficient_RN18', 'Addepalli2021Towards_RN18', 'Engstrom2019Robustness', 'Kang2021Stable', 'Bartoldson2024Adversarial_WRN-94-16']
Attacks:      ['Admix']
Ensemble:     True
Models Dir:   /kaggle/input/models/balajisb18/cifar-10/pytorch/default/1/models
Loading CIFAR-10 test data
Loaded 10000 samples, shape=torch.Size([10000, 3, 32, 32])

Loading ensemble surrogate (all surrogates combined)


/usr/local/lib/python3.12/dist-packages/torch/hub.py:247: UserWarning: You are about to download and run code from an untrusted repository. In a future release, this won't be allowed. To add the repository to your trusted list, change the command to load(..., trust_repo=False) and a command prompt will appear asking for an explicit confirmation of trust, or load(..., trust_repo=True), which will assume that the prompt is to be answered with 'yes'. You can also use load(..., trust_repo='check') which will only prompt for confirmation if the repo is not already trusted. This will eventually be the default behaviour
  _check_repo_is_trusted(


Downloading: "https://github.com/chenyaofo/pytorch-cifar-models/zipball/master" to /root/.cache/torch/hub/master.zip
Downloading: "https://github.com/chenyaofo/pytorch-cifar-models/releases/download/vgg/cifar10_vgg16_bn-6ee7ea24.pt" to /root/.cache/torch/hub/checkpoints/cifar10_vgg16_bn-6ee7ea24.pt


100%|██████████| 58.3M/58.3M [00:00<00:00, 144MB/s]
Using cache found in /root/.cache/torch/hub/chenyaofo_pytorch-cifar-models_master


Downloading: "https://github.com/chenyaofo/pytorch-cifar-models/releases/download/resnet/cifar10_resnet56-187c023a.pt" to /root/.cache/torch/hub/checkpoints/cifar10_resnet56-187c023a.pt


...

Downloading: "https://github.com/chenyaofo/pytorch-cifar-models/releases/download/repvgg/cifar10_repvgg_a2-09488915.pt" to /root/.cache/torch/hub/checkpoints/cifar10_repvgg_a2-09488915.pt


100%|██████████| 103M/103M [00:00<00:00, 151MB/s]



------------------------------------------------------------
Surrogate: Ensemble  |  Attack: Admix
------------------------------------------------------------


Generating adversarial examples: 100%|██████████| 2000/2000 [4:06:36<00:00,  7.40s/it]


Shape of adversarial images: torch.Size([10000, 3, 32, 32])
Adversarial examples generated in 14796.8s
Evaluating transfer to Chen2020Adversarial
Saved 263 fooled image pairs to /kaggle/working/results/fooled_images/Ensemble__Admix__Chen2020Adversarial/
Clean acc: 86.04%  Adv acc: 83.58%  ASR: 3.06%  Linf: 0.0314
Evaluating transfer to Rebuffi2021Fixing_70_16_cutmix_extra
Saved 218 fooled image pairs to /kaggle/working/results/fooled_images/Ensemble__Admix__Rebuffi2021Fixing_70_16_cutmix_extra/
Clean acc: 92.23%  Adv acc: 90.13%  ASR: 2.36%  Linf: 0.0314
Evaluating transfer to Wang2023Better_WRN-70-16
Saved 197 fooled image pairs to /kaggle/working/results/fooled_images/Ensemble__Admix__Wang2023Better_WRN-70-16/
Clean acc: 93.25%  Adv acc: 91.31%  ASR: 2.11%  Linf: 0.0314
Evaluating transfer to Sehwag2021Proxy_ResNest152
Saved 209 fooled image pairs to /kaggle/working/results/fooled_images/Ensemble__Admix__Sehwag2021Proxy_ResNest152/
Clean acc: 87.21%  Adv acc: 85.23%  ASR: 2.40%  Linf

In [9]:
import shutil
shutil.make_archive("download", 'zip', "/kaggle/working/results")


'/kaggle/working/download.zip'